In [13]:
import csv
import os

def create_interval_data_dict(xmin, xmax, sentence):
    return { 'xmin': float(xmin), 'xmax': float(xmax), 'text': sentence }

def print_interval_data_dict(idx, data):
    print(f"intervals [{idx}]:\n\t\txmin = {data['xmin']}\n\t\txmax = {data['xmax']}\n\t\ttext = \"{data['text']}\"")

def write_textgrid_file(intervals, output_file_path, total_xmax):
    print(f'Writing {output_file_path}')
    with open(output_file_path, 'w') as f:
        f.write('File type = "ooTextFile"\n')
        f.write('Object class = "TextGrid"\n')
        f.write('\n')
        f.write('xmin = 0\n')
        f.write(f'xmax = {str(float(total_xmax) + 0.001)}\n')
        f.write('tiers? <exists>\n')
        f.write('size = 1\n')
        f.write('item []:\n')
        f.write('    item [1]:\n')
        f.write('        class = "IntervalTier"\n')
        f.write('        name = "Intonational Unit"\n')
        f.write('        xmin = 0\n')
        f.write(f'        xmax = {str(float(total_xmax) + 0.001)}\n')
        f.write(f'        intervals: size = {len(intervals) + 1}\n')

        for idx, interval in enumerate(intervals):
            f.write(f'        intervals [{idx + 1}]:\n')
            f.write(f'            xmin = {interval["xmin"]}\n')
            f.write(f'            xmax = {interval["xmax"]}\n')
            f.write(f'            text = "{interval["text"]}"\n')
        
        if len(intervals) > 0:
            f.write(f'        intervals [{len(intervals) + 1}]:\n')
            f.write(f'            xmin = {intervals[-1]["xmax"]}\n')
            final_boundary = str(float(intervals[-1]["xmax"]) + 0.001)
            f.write(f'            xmax = {final_boundary}\n')
            f.write(f'            text = ""\n')

output_directory = 'textgrids'
os.makedirs(output_directory, exist_ok=True)

with open('spice_segmentation_run3.csv') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # Skip the header row
    iu_xmin = 0
    iu_xmax = 0
    intervals = []
    words = []
    prev_filename = None

    for row in reader:
        try:
            filename = row[1]
            xmin = float(row[2])
            xmax = float(row[3])
            text = row[4]
            is_iu_start_pred = row[5] == 'True'
        except IndexError:
            print(f"Skipping malformed row: {row}")
            continue

        if prev_filename is not None and prev_filename != filename:
            if len(intervals) > 0:
                write_textgrid_file(intervals, os.path.join(output_directory, f"{prev_filename}.TextGrid"), intervals[-1]['xmax'])
            iu_xmin = 0
            iu_xmax = 0
            intervals = []
            words = []

        prev_filename = filename

        if is_iu_start_pred:
            if len(intervals) == 0:
                prev_xmax = 0
            else:
                prev_xmax = intervals[-1]['xmax']

            if prev_xmax != iu_xmin:
                interval_data_dict = create_interval_data_dict(prev_xmax, iu_xmin, '')
                intervals.append(interval_data_dict)

            if len(words) > 0:
                interval_data_dict = create_interval_data_dict(iu_xmin, iu_xmax, ' '.join(words))
                intervals.append(interval_data_dict)
                words = []

            iu_xmin = xmin

        words.append(text)
        iu_xmax = xmax

    if len(intervals) > 0:
        write_textgrid_file(intervals, os.path.join(output_directory, f"{prev_filename}.TextGrid"), intervals[-1]['xmax'])

Writing textgrids/VF21A_Cantonese_I2_20190130.TextGrid
Writing textgrids/VF23B_Cantonese_I2_20190121.TextGrid
Writing textgrids/VF26A_Cantonese_I1_20190303.TextGrid
Writing textgrids/VF22A_Cantonese_I1_20181206.TextGrid
Writing textgrids/VF21B_Cantonese_I1_20190204.TextGrid
Writing textgrids/VF20B_Cantonese_I1_20181203.TextGrid
Writing textgrids/VF21C_Cantonese_I1_20190211.TextGrid
Writing textgrids/VF23C_Cantonese_I1_20190128.TextGrid
Writing textgrids/VF21D_Cantonese_I2_20190306.TextGrid


In [14]:
import pandas as pd

data = pd.read_csv('spice_segmentation_run3.csv')
data['file_name'].nunique()

9